# Cross Encoder Re-Ranking

## What is Cross Encoder Re-Ranking?

Cross Encoder Re-Ranking is a technique used in information retrieval and search systems to improve the relevance of search results. It's typically used as a **second stage** after an initial retrieval step.

## The Two-Stage Pipeline

1. **First Stage (Retrieval)**: Use a fast but less accurate method (like BM25 or bi-encoders) to retrieve a candidate set of documents (e.g., top 100)
2. **Second Stage (Re-Ranking)**: Use a more accurate but slower Cross Encoder to re-rank these candidates

![image-2.png](./76c2e575_image-2.png)

## How Cross Encoders Work

Unlike **bi-encoders** that encode query and document separately, a **Cross Encoder**:
- Takes the query and document as a **single input pair**
- Processes them together through a transformer model
- Outputs a relevance score directly

This allows the model to capture fine-grained interactions between query and document tokens, making it more accurate but computationally expensive.

![image.png](./76c2e575_image.png)

### Why Use Cross Encoders?

**Advantages:**
- **Higher accuracy**: Can capture complex interactions between query and document
- **Better semantic understanding**: The transformer can attend across both texts simultaneously

**Disadvantages:**
- **Computationally expensive**: Must process every query-document pair through the full model
- **Not suitable for large-scale retrieval**: Can't pre-compute document embeddings

### When to Use This Pattern?

Cross Encoder re-ranking is ideal when:
- You have a large corpus (millions of documents)
- You need high-quality top results
- You can afford two-stage processing
- The initial retrieval narrows down to a manageable set (10-1000 candidates)

### Practical Considerations

1. **Top-K Selection**: Typically re-rank 10-100 documents from the first stage
2. **Model Choice**: Smaller models (like MiniLM) balance speed and accuracy
3. **Batch Processing**: Process multiple pairs together for efficiency
4. **Caching**: Cache scores for frequently accessed documents

This two-stage approach combines the speed of traditional retrieval with the accuracy of deep learning, making it a popular choice in modern search systems!

# FAISS + Cross Encoder Re-Ranking Pattern

1. **FAISS (Dense Retrieval)**: Fast approximate nearest neighbor search using bi-encoders
2. **Cross Encoder**: Accurate re-ranking of top candidates

This is considered a **best practice** in production search systems and is used by companies like Pinecone, Weaviate, and many others.

## Why This Pattern Works So Well

### The Architecture

```
Query → Bi-Encoder → FAISS Index → Top-K Candidates → Cross Encoder → Final Ranked Results
        (fast)       (very fast)    (e.g., 100)        (accurate)      (e.g., 10)
```

### Key Benefits

1. **Scalability**: FAISS can handle millions/billions of vectors efficiently
2. **Speed**: Vector similarity search is extremely fast (milliseconds)
3. **Accuracy**: Cross Encoder refines the ranking for better relevance
4. **Cost-Effective**: Only run expensive Cross Encoder on a small subset

## Implementation

### Perfect Use Cases

1. **Large-Scale Search**: Millions of documents, need sub-second response times
2. **High Precision Requirements**: Top results must be highly relevant (e.g., legal, medical)
3. **Semantic Search**: Beyond keyword matching to understand user intent
4. **Recommendation Systems**: Finding similar items with refined ranking

### Real-World Examples

- **Pinecone/Weaviate**: Vector databases use this pattern internally
- **E-commerce**: Product search with semantic understanding
- **RAG Systems**: Retrieval-Augmented Generation for LLMs
- **Document Search**: Enterprise knowledge bases

## Alternative Patterns

- **BM25 + Cross Encoder**: Lexical retrieval + semantic re-ranking
- **Hybrid (BM25 + FAISS) + Cross Encoder**: Combine lexical and semantic first stage
- **Ensemble**: Multiple retrieval methods with Cross Encoder deciding final ranking

In [1]:
"""
Complete Search Pipeline: FAISS Dense Retrieval + Cross Encoder Re-Ranking
This is a production-grade pattern used in modern search systems

Installation:
pip install sentence-transformers faiss-cpu numpy
# Use faiss-gpu for GPU acceleration if available
"""

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder
from typing import List, Tuple

# ============================================================
# Setup Models and Data
# ============================================================

print("Loading models...")
# Bi-encoder for creating embeddings (used with FAISS)
bi_encoder = SentenceTransformer('all-MiniLM-L12-v2')

# Cross-encoder for re-ranking
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# Sample corpus (in production, this could be millions of documents)
corpus = [
    "Paris is the capital and most populous city of France, known for the Eiffel Tower.",
    "France is a country in Western Europe with rich history and culture.",
    "The Eiffel Tower is an iconic iron lattice tower located in Paris.",
    "London is the capital of the United Kingdom and a major global city.",
    "French cuisine is world-famous, including dishes like croissants and coq au vin.",
    "The Louvre Museum in Paris is the world's largest art museum.",
    "Berlin is the capital and largest city of Germany.",
    "Rome, the capital of Italy, is known for ancient Roman architecture.",
    "Madrid is the capital of Spain and home to the Royal Palace.",
    "The Seine river flows through the heart of Paris, France.",
    "Barcelona is a major city in Spain, famous for Gaudí's architecture.",
    "French wine regions like Bordeaux and Burgundy are renowned worldwide.",
    "The Palace of Versailles near Paris was the principal royal residence.",
    "Lyon is France's third-largest city, known as the gastronomic capital.",
    "Marseille is a major French port city on the Mediterranean coast.",
]

print(f"Corpus size: {len(corpus)} documents")

# ============================================================
# Stage 1: Build FAISS Index
# ============================================================

print("\n" + "="*60)
print("STAGE 1: Building FAISS Index")
print("="*60)

# Encode all documents into embeddings
print("Encoding documents with bi-encoder...")
corpus_embeddings = bi_encoder.encode(corpus, convert_to_numpy=True)

# Get embedding dimension
embedding_dim = corpus_embeddings.shape[1]
print(f"Embedding dimension: {embedding_dim}")

# Create FAISS index
# Using IndexFlatIP for inner product (cosine similarity with normalized vectors)
index = faiss.IndexFlatIP(embedding_dim)

# Normalize embeddings for cosine similarity
faiss.normalize_L2(corpus_embeddings)

# Add embeddings to index
index.add(corpus_embeddings)
print(f"Added {index.ntotal} vectors to FAISS index")

# ============================================================
# Stage 2: Search Function with Re-Ranking
# ============================================================

def search_with_faiss_and_rerank(
    query: str,
    top_k_retrieval: int = 10,
    top_k_final: int = 5
) -> List[Tuple[str, float, float]]:
    """
    Two-stage search pipeline:
    1. FAISS retrieves top_k_retrieval candidates
    2. Cross Encoder re-ranks to get top_k_final results
    
    Args:
        query: Search query string
        top_k_retrieval: Number of candidates to retrieve from FAISS
        top_k_final: Number of final results after re-ranking
    
    Returns:
        List of (document, faiss_score, rerank_score) tuples
    """
    
    # STAGE 1: Dense retrieval with FAISS
    print(f"\n🔍 Query: '{query}'")
    print(f"\nStage 1: FAISS retrieval (top {top_k_retrieval})...")
    
    # Encode query
    query_embedding = bi_encoder.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_embedding)
    
    # Search FAISS index
    faiss_scores, faiss_indices = index.search(query_embedding, top_k_retrieval)
    faiss_scores = faiss_scores[0]  # Get first (and only) query results
    faiss_indices = faiss_indices[0]
    
    # Get candidate documents
    candidates = [corpus[idx] for idx in faiss_indices]
    
    print("FAISS Results:")
    for i, (idx, score, doc) in enumerate(zip(faiss_indices, faiss_scores, candidates), 1):
        print(f"  {i}. [FAISS: {score:.4f}] {doc}")
    
    # STAGE 2: Re-ranking with Cross Encoder
    print(f"\nStage 2: Cross Encoder re-ranking...")
    
    # Create query-document pairs
    pairs = [[query, doc] for doc in candidates]
    
    # Get re-ranking scores
    rerank_scores = cross_encoder.predict(pairs)
    
    # Combine results with both scores
    combined_results = list(zip(candidates, faiss_scores, rerank_scores))
    
    # Sort by re-ranking scores
    combined_results.sort(key=lambda x: x[2], reverse=True)
    
    # Return top_k_final results
    return combined_results[:top_k_final]

# ============================================================
# Example Usage
# ============================================================

print("\n" + "="*60)
print("SEARCH EXAMPLES")
print("="*60)

# Example 1: Capital of France
results = search_with_faiss_and_rerank(
    query="What is the capital of France?",
    top_k_retrieval=10,
    top_k_final=5
)

print("\n✅ Final Re-Ranked Results:")
for rank, (doc, faiss_score, rerank_score) in enumerate(results, 1):
    improvement = "🔺" if rerank_score > faiss_score else "🔻"
    print(f"{rank}. {improvement} [FAISS: {faiss_score:.4f} → Rerank: {rerank_score:.4f}]")
    print(f"   {doc}\n")

# ============================================================
# Compare Rankings: With vs Without Re-Ranking
# ============================================================

print("\n" + "="*60)
print("COMPARISON: Impact of Re-Ranking")
print("="*60)

query = "famous museums and art in Paris"

# Without re-ranking (just FAISS)
query_embedding = bi_encoder.encode([query], convert_to_numpy=True)
faiss.normalize_L2(query_embedding)
faiss_scores, faiss_indices = index.search(query_embedding, 5)
faiss_only_results = [(corpus[idx], faiss_scores[0][i]) 
                      for i, idx in enumerate(faiss_indices[0])]

# With re-ranking
results = search_with_faiss_and_rerank(query, top_k_retrieval=10, top_k_final=5)

print("\n📊 FAISS + Cross Encoder Re-Ranking:")
for rank, (doc, faiss_score, rerank_score) in enumerate(results, 1):
    print(f"{rank}. [FAISS: {faiss_score:.4f} → Rerank: {rerank_score:.4f}] {doc}")

# ============================================================
# Performance Metrics
# ============================================================

print("\n" + "="*60)
print("PERFORMANCE ANALYSIS")
print("="*60)

import time

# Measure FAISS search time
query = "landmarks in France"
query_embedding = bi_encoder.encode([query], convert_to_numpy=True)
faiss.normalize_L2(query_embedding)

start = time.time()
for _ in range(100):
    faiss_scores, faiss_indices = index.search(query_embedding, 10)
faiss_time = (time.time() - start) / 100

print(f"\n⚡ FAISS search (avg over 100 runs): {faiss_time*1000:.2f} ms")

# Measure Cross Encoder time
candidates = [corpus[idx] for idx in faiss_indices[0]]
pairs = [[query, doc] for doc in candidates]

start = time.time()
rerank_scores = cross_encoder.predict(pairs)
rerank_time = time.time() - start

print(f"🔄 Cross Encoder re-ranking (10 docs): {rerank_time*1000:.2f} ms")
print(f"📊 Total pipeline time: {(faiss_time + rerank_time)*1000:.2f} ms")

print("\n💡 Insights:")
print(f"   - FAISS handles the heavy lifting (searching {len(corpus)} docs)")
print(f"   - Cross Encoder only processes 10 candidates")
print(f"   - This scales to millions of documents efficiently!")

# ============================================================
# Production-Ready Class
# ============================================================

class HybridSearchEngine:
    """
    Production-ready search engine combining FAISS and Cross Encoder
    """
    
    def __init__(self, bi_encoder_name: str, cross_encoder_name: str):
        self.bi_encoder = SentenceTransformer(bi_encoder_name)
        self.cross_encoder = CrossEncoder(cross_encoder_name)        
        self.index = None
        self.corpus = []
        
    def build_index(self, documents: List[str]):
        """Build FAISS index from documents"""
        self.corpus = documents
        embeddings = self.bi_encoder.encode(documents, convert_to_numpy=True)
        
        embedding_dim = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(embedding_dim)
        
        faiss.normalize_L2(embeddings)
        self.index.add(embeddings)
        
        print(f"✅ Index built with {len(documents)} documents")
        
    def search(self, query: str, top_k: int = 5, 
               retrieval_k: int = 20) -> List[Tuple[str, float]]:
        """Search with FAISS + Cross Encoder re-ranking"""
        
        # Stage 1: FAISS retrieval
        query_emb = self.bi_encoder.encode([query], convert_to_numpy=True)
        faiss.normalize_L2(query_emb)
        
        _, indices = self.index.search(query_emb, retrieval_k)
        candidates = [self.corpus[idx] for idx in indices[0]]
        
        # Stage 2: Cross Encoder re-ranking
        pairs = [[query, doc] for doc in candidates]
        scores = self.cross_encoder.predict(pairs)
        
        # Sort and return top-k
        results = list(zip(candidates, scores))
        results.sort(key=lambda x: x[1], reverse=True)
        
        return results[:top_k]

# Example usage of the class
print("\n" + "="*60)
print("PRODUCTION-READY CLASS EXAMPLE")
print("="*60)

engine = HybridSearchEngine(
    bi_encoder_name='sentence-transformers/all-MiniLM-L12-v2',
    cross_encoder_name='cross-encoder/ms-marco-MiniLM-L-6-v2'
)

engine.build_index(corpus)

results = engine.search("art museums in Paris", top_k=3)
print("\n🎯 Search Results:")
for rank, (doc, score) in enumerate(results, 1):
    print(f"{rank}. [Score: {score:.4f}] {doc}")

Loading models...


/home/admin/_github/massimodipaolo/ai-crash-course/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:283: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  warnings.warn(


Corpus size: 15 documents

STAGE 1: Building FAISS Index
Encoding documents with bi-encoder...
Embedding dimension: 384
Added 15 vectors to FAISS index

SEARCH EXAMPLES

🔍 Query: 'What is the capital of France?'

Stage 1: FAISS retrieval (top 10)...
FAISS Results:
  1. [FAISS: 0.5549] France is a country in Western Europe with rich history and culture.
  2. [FAISS: 0.5033] Paris is the capital and most populous city of France, known for the Eiffel Tower.
  3. [FAISS: 0.4157] Marseille is a major French port city on the Mediterranean coast.
  4. [FAISS: 0.4096] Lyon is France's third-largest city, known as the gastronomic capital.
  5. [FAISS: 0.4038] French wine regions like Bordeaux and Burgundy are renowned worldwide.
  6. [FAISS: 0.3737] The Palace of Versailles near Paris was the principal royal residence.
  7. [FAISS: 0.3720] The Seine river flows through the heart of Paris, France.
  8. [FAISS: 0.3177] Madrid is the capital of Spain and home to the Royal Palace.
  9. [FAISS: 0.31

## The Key Insight: 

> Cross Encoders is a transformer model that takes as input the concatenation of the query and document text, and outputs a relevance score directly, rather than comparing pre-computed embeddings.

### What Actually Happens

1. **Bi-Encoder (for FAISS)**: 
   - Encodes query → vector
   - Encodes document → vector  
   - Compares vectors with cosine similarity
   - **Outputs: similarity score**

2. **Cross Encoder (for re-ranking)**:
   - Takes `[query, document]` as **raw text** concatenated
   - Processes through transformer (like BERT)
   - Outputs: single relevance score from classification head (single scalar value)

### The Token Flow
```
Input:  "[CLS] How do I fix a flat tire? [SEP] You need a jack and spare tire. [SEP]"
         ↓
Transformer Layers (with cross-attention between query & doc tokens)
         ↓
[CLS] token representation (captures the relationship)
         ↓
Linear Classification Head
         ↓
Output: 0.87 (relevance score)   
```

In [4]:
print(engine.cross_encoder.model)
print('-'*40)
print(engine.cross_encoder.model.config)
print('-'*40)
num_params = sum(p.numel() for p in engine.cross_encoder.model.parameters())
print(f"Parameters: {num_params}")

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 384, padding_idx=0)
      (position_embeddings): Embedding(512, 384)
      (token_type_embeddings): Embedding(2, 384)
      (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-5): 6 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=384, out_features=384, bias=True)
              (key): Linear(in_features=384, out_features=384, bias=True)
              (value): Linear(in_features=384, out_features=384, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=384, out_features=384, bias=True)
              (LayerNorm): LayerNorm((384,), eps=1e-1